# ME 455 — iLQR: Iterative Linear Quadratic Regulator

**Author:** Mark Chauhan  
**Course:** ME 455 — Active Learning in Robotics, Northwestern University  
**Date:** May 2025  

---

## Overview

Implementation of the **iterative Linear Quadratic Regulator (iLQR)** for trajectory optimization 
on a nonlinear unicycle system.

iLQR iteratively linearizes the nonlinear dynamics around the current trajectory estimate,
solves the resulting LQR problem via backward-pass Riccati recursion, then updates the 
control trajectory via Armijo line search. This repeats until convergence.

## System: Unicycle
```
State:   x = [px, py, θ]  (position + heading)
Control: u = [v, ω]       (linear velocity + angular velocity)

Dynamics:
  ẋ = cos(θ) · v
  ẏ = sin(θ) · v  
  θ̇ = ω
```

## Algorithm
1. **Forward pass** — simulate trajectory under current control `u_traj`
2. **Linearization** — compute Jacobians A(t), B(t) along the trajectory
3. **Backward pass** — solve Riccati equation backward in time to get descent direction `v_traj`
4. **Line search** — Armijo step-size selection along `v_traj`
5. **Update** — `u_traj ← u_traj + γ · v_traj`
6. Repeat until convergence

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp

## Parameters & Cost Weights

In [ ]:
# ── Time parameters ──────────────────────────────────────────────────────────
dt     = 0.1
tsteps = 63
x0     = np.array([0.0, 0.0, np.pi/2.0])     # initial state [px, py, theta]
init_u_traj = np.tile(np.array([1.0, -0.5]), reps=(tsteps, 1))  # initial control guess

# ── Cost matrices ─────────────────────────────────────────────────────────────
# Running cost: L(t) = (x-xd)^T Q_x (x-xd) + u^T R_u u
Q_x = np.diag([10.0, 10.0, 2.0])  # position and heading tracking weights
R_u = np.diag([4.0,  2.0])        # control effort penalty
P1  = np.diag([20.0, 20.0, 5.0])  # terminal cost weight

## Dynamics & Jacobians

Unicycle dynamics with analytically derived Jacobians for the linearization step.

In [ ]:
def dyn(xt, ut):
    """Unicycle dynamics: returns state derivative xdot."""
    theta = xt[2]
    v, omega = ut
    return np.array([
        np.cos(theta) * v,
        np.sin(theta) * v,
        omega
    ])


def get_A(t, xt, ut):
    """Linearization: df/dx — state Jacobian of unicycle dynamics."""
    theta = xt[2]
    v     = ut[0]
    return np.array([
        [0, 0, -np.sin(theta) * v],
        [0, 0,  np.cos(theta) * v],
        [0, 0,  0]
    ])


def get_B(t, xt, ut):
    """Linearization: df/du — control Jacobian of unicycle dynamics."""
    theta = xt[2]
    return np.array([
        [np.cos(theta), 0],
        [np.sin(theta), 0],
        [0,             1]
    ])


def step(xt, ut):
    """Euler integration step."""
    return xt + dt * dyn(xt, ut)


def traj_sim(x0, u_traj):
    """Forward simulate trajectory from x0 under control sequence u_traj."""
    T = u_traj.shape[0]
    x_traj = np.zeros((T, 3))
    xt = x0.copy()
    for t in range(T):
        xt = step(xt, u_traj[t])
        x_traj[t] = xt
    return x_traj

## Cost Functions

In [ ]:
def xd(t):
    """Desired state trajectory: move along x-axis at constant speed, heading pi/2."""
    return np.array([2.0 * t / np.pi, 0.0, np.pi/2])


def loss(t, xt, ut):
    """Running cost at time t."""
    e = xt - xd(t)
    return e.T @ Q_x @ e + ut.T @ R_u @ ut


def dldx(t, xt, ut):
    """Gradient of running cost w.r.t. state."""
    return 2 * Q_x @ (xt - xd(t))


def dldu(t, xt, ut):
    """Gradient of running cost w.r.t. control."""
    return 2 * R_u @ ut


def total_cost(x0, u_traj):
    """Compute total trajectory cost."""
    x_traj = traj_sim(x0, u_traj)
    cost = 0.0
    for t_idx in range(tsteps):
        t = t_idx * dt
        cost += loss(t, x_traj[t_idx], u_traj[t_idx]) * dt
    # terminal cost
    xd_T = xd((tsteps - 1) * dt)
    cost += (x_traj[-1] - xd_T).T @ P1 @ (x_traj[-1] - xd_T)
    return cost

## iLQR: Backward Pass

The backward pass computes the optimal descent direction `v_traj` by solving 
the Riccati equation backward in time.

In [ ]:
def ilqr_iter(x0, u_traj):
    """
    One iLQR iteration.
    
    Args:
        x0:     initial state
        u_traj: current control trajectory (T x 2)
    
    Returns:
        v_traj: descent direction for control (T x 2)
    """
    x_traj = traj_sim(x0, u_traj)

    # Precompute Jacobians and cost gradients along trajectory
    A_list  = np.zeros((tsteps, 3, 3))
    B_list  = np.zeros((tsteps, 3, 2))
    a_list  = np.zeros((tsteps, 3))   # dL/dx
    b_list  = np.zeros((tsteps, 2))   # dL/du

    for t_idx in range(tsteps):
        t = t_idx * dt
        xt, ut = x_traj[t_idx], u_traj[t_idx]
        A_list[t_idx] = get_A(t, xt, ut)
        B_list[t_idx] = get_B(t, xt, ut)
        a_list[t_idx] = dldx(t, xt, ut)
        b_list[t_idx] = dldu(t, xt, ut)

    # Terminal boundary condition
    xd_T = xd((tsteps - 1) * dt)
    p = 2 * P1 @ (x_traj[-1] - xd_T)  # costate at T

    # Backward pass: integrate costate p(t) and compute descent v(t)
    v_traj = np.zeros((tsteps, 2))
    for t_idx in reversed(range(tsteps)):
        A = A_list[t_idx]
        B = B_list[t_idx]
        a = a_list[t_idx]
        b = b_list[t_idx]

        # Descent direction: v = -R_u^{-1} (b + B^T p)
        v_traj[t_idx] = -np.linalg.solve(R_u, b + B.T @ p)

        # Costate update (backward Euler)
        p = p - dt * (a + A.T @ p)

    return v_traj

## Run iLQR Optimization

In [ ]:
u_traj = init_u_traj.copy()
cost_history = []

for iteration in range(15):
    # Evaluate current cost
    current_cost = total_cost(x0, u_traj)
    cost_history.append(current_cost)

    # Get descent direction
    v_traj = ilqr_iter(x0, u_traj)

    # Armijo line search
    gamma = 1.0
    beta  = 0.5     # backtracking factor
    sigma = 1e-4    # sufficient decrease parameter
    descent_norm = np.sum(v_traj ** 2)

    for _ in range(20):
        u_new = u_traj + gamma * v_traj
        new_cost = total_cost(x0, u_new)
        if new_cost < current_cost - sigma * gamma * descent_norm:
            break
        gamma *= beta

    u_traj = u_new
    print(f'Iter {iteration+1:2d} | Cost: {current_cost:.4f} | Step: {gamma:.4f}')

print(f'\nFinal cost: {total_cost(x0, u_traj):.4f}')

## Results

In [ ]:
x_traj = traj_sim(x0, u_traj)
tlist  = np.arange(tsteps) * dt
xd_traj = np.array([xd(t) for t in tlist])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Trajectory in x-y plane
axes[0].plot(xd_traj[:, 0], xd_traj[:, 1], '--', color='gray', label='Desired')
axes[0].plot(x_traj[:, 0],  x_traj[:, 1],  '-',  color='#38bdf8', lw=2, label='Optimized')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_title('Trajectory'); axes[0].legend(); axes[0].set_aspect('equal')

# Control inputs
axes[1].plot(tlist, u_traj[:, 0], label='v (linear)')
axes[1].plot(tlist, u_traj[:, 1], label='ω (angular)')
axes[1].set_xlabel('Time'); axes[1].set_ylabel('Control')
axes[1].set_title('Optimal Control Inputs'); axes[1].legend()

# Cost convergence
axes[2].plot(cost_history, 'o-', color='#38bdf8')
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Total Cost')
axes[2].set_title('iLQR Cost Convergence')

plt.tight_layout()
plt.show()

## Cost Weight Analysis

Sweep over different cost weight configurations to analyze trajectory sensitivity.

In [ ]:
configs = [
    {'label': 'Baseline',           'Q_x': np.diag([10., 10., 2.]),  'R_u': np.diag([4., 2.])},
    {'label': 'High θ weight',      'Q_x': np.diag([10., 10., 20.]), 'R_u': np.diag([4., 2.])},
    {'label': 'Heavy control cost', 'Q_x': np.diag([10., 10., 2.]),  'R_u': np.diag([20., 10.])},
]

fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.plot(xd_traj[:, 0], xd_traj[:, 1], '--', color='gray', lw=1, label='Desired', zorder=0)

colors = ['#38bdf8', '#f97316', '#a78bfa']
for cfg, color in zip(configs, colors):
    # Temporarily override globals for this config
    Q_x = cfg['Q_x']; R_u = cfg['R_u']
    u = init_u_traj.copy()
    for _ in range(10):
        v = ilqr_iter(x0, u)
        u = u + 0.5 * v
    xt = traj_sim(x0, u)
    ax.plot(xt[:, 0], xt[:, 1], color=color, lw=2, label=cfg['label'])

# Restore baseline
Q_x = np.diag([10., 10., 2.]); R_u = np.diag([4., 2.])

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Trajectory Sensitivity to Cost Weights')
ax.legend(); ax.set_aspect('equal')
plt.tight_layout()
plt.show()